# 深度學習框架應用（大學部）｜主題 6：經典 CNN 架構與遷移學習

這本筆記本配合投影片《主題 6：經典 CNN 架構與遷移學習》。前兩題拆開經典架構（殘差區塊、VGG 和 ResNet 的參數），第三題做資料增強，後兩題用預訓練的 ResNet-18 做兩階段遷移學習。**練習一～五，每題 20%，合計 100%**。

| 練習 | 題目 | 投影片（練習頁） | 觀念在哪一頁 |
|---|---|---|---|
| 一 | 手刻殘差區塊（形狀與參數量） | 第 115 頁 | 第 34、37、39 頁 |
| 二 | VGG-16 頭重腳輕？再算瓶頸塊 | 第 116 頁 | 第 22～24、28、37 頁 |
| 三 | 資料增強：同一張圖變出 8 種樣子 | 第 117 頁 | 第 17～19 頁 |
| 四 | 遷移學習①：凍結骨幹，只訓練新的分類頭 | 第 118 頁 | 第 61、73、78 頁 |
| 五 | 遷移學習②：解凍 layer4，兩組學習率微調 | 第 119 頁 | 第 84、87、92 頁 |

**怎麼用**

1. 用私人 Google 帳號登入（不使用學校帳號），按上方的「複製到雲端硬碟」（或「檔案 › 在雲端硬碟中儲存複本」）另存一份自己的複本，之後都改自己那份。**沒有另存就直接寫，分頁一關就全部不見。** 存好後先把檔名改成 `深度學習_W06_學號_姓名.ipynb`（左上角點檔名就能改）。
2. 先跑「環境準備」，再由上往下做；標「準備（直接執行）」的格子不用改，直接跑。每一題只有一格標 ✏️ 的程式格：把每個 `___` 換成答案再執行，提示寫在同一行後面的註解。還沒把 `___` 換掉就執行，會出現紅字錯誤（例如 TypeError、AttributeError），錯誤訊息裡箭頭 `---->` 指的那一行就是還沒填的空格，填好再跑一次就好。
3. 有 GPU 就用（「執行階段 › 變更執行階段類型」選 T4 GPU），沒有也沒關係：每一題用 CPU 都能在幾分鐘內跑完。練習四第一次執行會下載 ResNet-18 的預訓練權重（約 45 MB）。
4. 練習五要接在練習四後面跑（模型要先經過階段一）。想重來，從「練習四、五：準備」那一格往下重跑。
5. **每題做完就截一張整個螢幕的圖（截圖 ①～⑤）**，要看得到身份指紋：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間，以及這一題的輸出。**看不到身份指紋，那一題不計分。**

**繳交**：做完照最後一格「繳交前檢查」，把**分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤）**交到 Flipclass 作業區。

## 環境準備

五題都會用到下面這些套件和三個小工具（`count` 數參數、`conv3` 是 3×3 卷積、`proj` 是跳接上的 1×1 投影），先跑這一格。

In [ ]:
# ===== 環境準備（直接執行）=====
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torchvision.transforms.v2 as T
from torchvision import datasets, models
from torch.utils.data import DataLoader, Subset
dev = 'cuda' if torch.cuda.is_available() else 'cpu'   # 有 GPU 就用；沒有，CPU 也跑得完
print('PyTorch：', torch.__version__, '｜裝置：', dev)

BN = nn.BatchNorm2d                     # 批次正規化（Batch Normalization），名字太長，取個小名

def count(m):                           # 數一個模型裡「可以學」的參數有幾個
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def conv3(cin, cout, s):                # 3×3 卷積：padding=1 邊長不變，s=2 時邊長減半；後面接 BN，所以不加偏移
    return nn.Conv2d(cin, cout, 3, s, padding=1, bias=False)

def proj(cin, cout, s):                 # 跳接上的 1×1 投影：形狀對不上時，把 x 變成和主路一樣的形狀
    return nn.Sequential(nn.Conv2d(cin, cout, 1, s, bias=False), BN(cout))

## 練習一：手刻殘差區塊（形狀與參數量） (20%)

> 題目在投影片**第 115 頁**｜觀念在第 34、37、39 頁

**情境**：主路是「3×3 卷積 → BN → ReLU → 3×3 卷積 → BN」，跳接把輸入 x 直接送到加號；形狀對不上時，跳接要加一個 1×1 卷積。這題自己寫一個殘差區塊，再和 torchvision 官方 ResNet-18 的區塊對答案。

**這題在練**：說得出殘差區塊的兩條路，知道什麼時候跳接要用 1×1 投影，並用形狀和參數量驗證自己寫的和官方一樣

**步驟**

1. 先執行「準備」那一格：它會印出官方 `layer1[0]`、`layer2[0]` 的輸出形狀和參數量，等一下拿來對答案
2. 填好 ✏️ 格的空格：第二個 BN 的通道數、投影的步幅、forward 裡加上跳接、第二個區塊的步幅（邊長減半）
3. 執行後比一比：兩行輸出的形狀和參數量，要和官方印出來的完全一樣

**要繳交**

- 截圖 ①：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習一、二：準備（直接執行，不用改）=====
torch.manual_seed(0)
x = torch.randn(2, 64, 56, 56)          # 假的特徵圖：2 張、64 個通道、56×56（ResNet-18 layer1 的大小）
tv = models.resnet18(weights=None)      # torchvision 官方的 ResNet-18：只拿架構來對答案，不下載權重
img = torch.randn(1, 3, 224, 224)       # 一張假的 224×224 彩色照片，練習二用
for name in ['layer1', 'layer2']:
    blk = getattr(tv, name)[0]          # 這一段的第 1 個殘差區塊
    print(f'官方 {name}[0]：輸出 {tuple(blk(x).shape)}，參數 {count(blk):,} 個')

In [ ]:
# ✏️ 練習一：手刻殘差區塊，驗證形狀與參數量
class Block(nn.Module):
    def __init__(self, cin, cout, s=1):
        super().__init__()
        self.main = nn.Sequential(
            conv3(cin, cout, s), BN(cout), nn.ReLU(),
            conv3(cout, cout, 1), BN(___))    # ✏️ 通道數
        self.skip = nn.Identity()  # 形狀不變：照抄
        if s != 1 or cin != cout:             # 形狀會變：
            self.skip = proj(cin, cout, ___)  # ✏️ 一起縮
    def forward(self, x):
        return (self.main(x) + self.___(x)).relu()  # ✏️
for b in [Block(64, 64), Block(64, 128, ___)]:  # ✏️ 步幅
    print(tuple(b(x).shape), f'{count(b):,}')

#### 📸 截圖 ①

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：「準備」印出的官方兩行，以及 ✏️ 格印出的兩行（形狀和參數量）

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 第二個區塊 Block(64, 128, s=2) 的跳接，為什麼一定要用 1×1 投影？**

（在這裡寫）

**2. 230,144 個參數是怎麼來的？（提示：兩個 3×3 卷積、一個 1×1 卷積、三個 BN）**

（在這裡寫）

## 練習二：VGG-16 頭重腳輕？再算瓶頸塊 (20%)

> 題目在投影片**第 116 頁**｜觀念在第 22～24、28、37 頁

**情境**：VGG-16 只用 3×3 卷積堆疊，最後接三層全連接；ResNet-18 用全域平均池化，只剩一層 fc。這題數一數兩個模型的「頭」各占多少參數，再算 ResNet-50 用的瓶頸塊省了多少。

**這題在練**：用 `count()` 數參數，親眼看到全連接的頭有多重；知道 VGG 的卷積段把 224 的照片縮成 7×7；算出瓶頸塊和兩個 3×3 的參數差多少

**步驟**

1. VGG 的頭叫 `classifier`、卷積段叫 `features`；ResNet-18 的頭叫 `fc`（練習一準備格的 `tv` 就是 ResNet-18）
2. 印出兩個模型的全部參數、頭的參數和占比，再讓一張 224×224 的照片走過 VGG 的卷積段，看形狀
3. 瓶頸塊：先用 1×1 把 256 個通道降到 64，做 3×3，再用 1×1 升回 256；和兩個 256 通道的 3×3 比一比

**要繳交**

- 截圖 ②：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習二：VGG-16 頭重腳輕？再算瓶頸塊
vgg = models.vgg16(weights=None)         # 只要架構
for name, m, h in [('VGG-16', vgg, vgg.___),  # ✏️
                   ('ResNet-18', tv, tv.___)]:  # ✏️
    t, h = count(m), count(h)            # 全部、頭
    print(f'{name}：全部 {t:,}，頭 {h:,}（{h / t:.1%}）')
print('池化五次後', tuple(vgg.___(img).shape))  # ✏️
one = conv3(256, 256, 1)  # 一個 256 通道的 3×3
neck = nn.Sequential(
    nn.Conv2d(256, ___, 1, bias=False),  # ✏️ 降到 64
    conv3(64, 64, 1),
    nn.Conv2d(64, 256, 1, bias=False))   # 升回 256
print('兩個 3×3：', f'{2 * count(one):,}')
print('瓶頸塊：', f'{count(neck):,}')

#### 📸 截圖 ②

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：四行結果（兩個模型的參數與占比、池化五次後的形狀、兩個 3×3 和瓶頸塊的參數）

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. VGG-16 的頭為什麼那麼重？第一層全連接的 25,088 是怎麼來的？**

（在這裡寫）

**2. 瓶頸塊的 69,632 怎麼算？比兩個 3×3 省了大約幾倍？**

（在這裡寫）

## 練習三：資料增強：同一張圖變出 8 種樣子 (20%)

> 題目在投影片**第 117 頁**｜觀念在第 17～19 頁

**情境**：Fashion-MNIST 的第 0 張是一隻短靴。訓練時讓每一張照片每次都「長得有點不一樣」（翻一下、轉一點、裁一點），模型就比較不會死背。

**這題在練**：用 `torchvision.transforms.v2` 組出訓練用的隨機增強，看同一張圖變出 8 種樣子；並分清楚訓練用的會變、驗證用的不能變

**步驟**

1. 先執行「準備資料」那一格（第一次會下載 Fashion-MNIST），它也定義了 `to_tensor` 和畫圖的 `show_grid`
2. 填好三個空格：左右翻的機率、RandomResizedCrop 至少留下原圖的幾成、驗證用的前處理
3. 看 8 張圖有什麼不一樣，再看「兩次一樣嗎？」那一行：訓練用的每次都不同，驗證用的每次都一樣

> 隨機增強只放在訓練集；驗證、測試只做 `to_tensor`（後面練習四還會加 Normalize），不加任何隨機。

**要繳交**

- 截圖 ③：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習三～五：準備資料（直接執行；第一次會下載約 30 MB）=====
raw = datasets.FashionMNIST('data', train=True, download=True)        # 訓練用 6 萬張（PIL 灰階圖，28×28）
datasets.FashionMNIST('data', train=False, download=True)             # 測試用 1 萬張，先下載好
print('訓練集：', len(raw), '張｜10 類：', raw.classes)
to_tensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])   # PIL 圖 → 0～1 的張量

def show_grid(imgs):                    # 把一疊 (N, 1, H, W) 的圖排成一列畫出來
    fig, axs = plt.subplots(1, len(imgs), figsize=(1.6 * len(imgs), 1.8))
    for k, (ax, im) in enumerate(zip(axs, imgs)):
        ax.imshow(im[0], cmap='gray'); ax.set_title(f'#{k}'); ax.axis('off')
    plt.show()

In [ ]:
# ✏️ 練習三：同一張圖，資料增強變出 8 種樣子
torch.manual_seed(0)  # 固定亂數，結果才重現得出來
pic, label = raw[0]                 # 第 0 張：PIL 灰階圖
train_tf = T.Compose([
    T.RandomHorizontalFlip(p=___),  # ✏️ 一半機率翻
    T.RandomRotation(degrees=15),   # 轉 15 度以內
    T.RandomResizedCrop(28, scale=(___, 1.0)),  # ✏️ 八成
    to_tensor])                     # 最後才轉成張量
outs = torch.stack([train_tf(pic) for _ in range(8)])
print(raw.classes[label], '→', tuple(outs.shape))
a = torch.equal(train_tf(pic), train_tf(pic))  # 訓練用
b = torch.equal(___(pic), to_tensor(pic))  # ✏️ 驗證
print('兩次一樣嗎？訓練用：', a, '｜驗證用：', b)
show_grid(outs)

#### 📸 截圖 ③

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：印出的兩行（類別名稱與 8 張疊起來的形狀、兩次一樣嗎），以及 8 張圖

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 為什麼驗證和測試不能加隨機的資料增強？**

（在這裡寫）

**2. 如果資料是手寫數字或文字，可以用左右翻轉嗎？為什麼？**

（在這裡寫）

## 練習四：遷移學習①：凍結骨幹，只訓練新的分類頭 (20%)

> 題目在投影片**第 118 頁**｜觀念在第 61、73、78 頁

**情境**：ResNet-18 已經用 ImageNet 的 100 多萬張照片學會「怎麼看東西」。我們只有 2,000 張衣服照片，所以骨幹全部凍住、把最後的 fc 換成 10 類的新頭，只訓練這個頭。

**這題在練**：載入預訓練模型、換分類頭、再搬一次裝置、凍結骨幹，並用可訓練參數量確認只剩分類頭在學（這是兩階段遷移學習的階段一）

**步驟**

1. 先執行「練習四、五：準備」：它印出官方前處理（mean、std）、做好 DataLoader（訓練 2,000 張、驗證 1,000 張），並定義跑一輪的 `run()`
2. 填好空格：新 fc 的輸入數、換完頭要做的事、凍結全部參數、再只打開 fc
3. 看「可訓練參數」那一行對不對，再看兩輪的 train／val 正確率

> 灰階照片用 `Grayscale(3)` 複製成三個通道；為了 CPU 也跑得完，邊長用 96（官方是 224），但 mean、std 一定照預訓練時的。

**要繳交**

- 截圖 ④：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習四、五：準備（直接執行，不用改）=====
weights = models.ResNet18_Weights.IMAGENET1K_V1        # ImageNet 預訓練的第一版權重（練習四才會真的下載）
prep = weights.transforms()                            # 官方前處理：裡面記著尺寸、mean、std
print(prep)
SIZE = 96                                              # 為了 CPU 也跑得完，邊長用 96（官方是 224）
tf = T.Compose([T.Grayscale(num_output_channels=3),    # 灰階複製成 3 個通道：ResNet 吃 RGB
                T.Resize(SIZE), to_tensor,
                T.Normalize(prep.mean, prep.std)])     # mean、std 一定照預訓練時的
train_set = datasets.FashionMNIST('data', train=True, transform=tf)
test_set = datasets.FashionMNIST('data', train=False, transform=tf)
g = torch.Generator().manual_seed(0)                   # 固定洗牌順序
train_dl = DataLoader(Subset(train_set, range(0, 2000)), batch_size=64, shuffle=True, generator=g)  # 訓練 2,000 張
val_dl = DataLoader(Subset(train_set, range(2000, 3000)), batch_size=128)   # 驗證 1,000 張：挑最好的那一版
test_dl = DataLoader(Subset(test_set, range(0, 1000)), batch_size=128)      # 測試 1,000 張：最後才用
xs, ys = next(iter(val_dl))
print('一批照片：', tuple(xs.shape), '｜一批標籤：', tuple(ys.shape))

def run(model, dl, opt=None):
    """跑一輪：有給 opt 就訓練，沒給就只評估。回傳 (平均 loss, 正確率)。"""
    train = opt is not None
    model.train(train)
    for m in model.modules():                          # 凍住的 BN 一律維持 eval：統計量才不會偷偷變
        if isinstance(m, nn.BatchNorm2d) and not m.weight.requires_grad:
            m.eval()
    loss_fn, tot, right, n = nn.CrossEntropyLoss(), 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in dl:
            xb, yb = xb.to(dev), yb.to(dev)
            out = model(xb)
            loss = loss_fn(out, yb)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(yb); right += (out.argmax(1) == yb).sum().item(); n += len(yb)
    return tot / n, right / n

torch.manual_seed(0)                                   # 讓新分類頭的起點固定

In [ ]:
# ✏️ 練習四：凍結骨幹，只訓練新的分類頭（階段一）
model = models.resnet18(weights=weights)  # 載入預訓練權重
model.fc = nn.Linear(model.fc.___, 10)  # ✏️ 輸入
model = model.___(dev)       # ✏️ 換完頭再搬一次
for p in model.parameters():
    p.requires_grad = ___    # ✏️ 全部先凍住
for p in model.fc.parameters():
    p.requires_grad = ___    # ✏️ 只打開頭
print('可訓練參數：', f'{count(model):,}')
opt = torch.optim.AdamW(model.fc.parameters(), lr=1e-3)
for ep in range(2):
    tr, va = run(model, train_dl, opt), run(model, val_dl)
    print(f'ep{ep}  train {tr[1]:.3f}  val {va[1]:.3f}')

#### 📸 截圖 ④

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：可訓練參數那一行，以及兩輪的 train／val 正確率

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 可訓練參數 5,130 是怎麼算的？大約占整個模型的多少？**

（在這裡寫）

**2. 為什麼換完頭之後要再 .to(dev) 一次？**

（在這裡寫）

## 練習五：遷移學習②：解凍 layer4，兩組學習率微調 (20%)

> 題目在投影片**第 119 頁**｜觀念在第 84、87、92 頁

**情境**：階段一已經把新的頭訓練得差不多了。階段二再把骨幹最後一段 layer4 解凍，用很小的學習率輕輕調，讓特徵更貼近衣服照片；分類頭的學習率大 10 倍。

**這題在練**：用兩組學習率（骨幹小、頭大）微調，並在驗證正確率創新高時存下最好的那一版

**步驟**

1. 確定練習四已經跑完（模型要先經過階段一）
2. 填好空格：要解凍的那一段、分類頭的學習率（1e-4，是骨幹 1e-5 的 10 倍）、存檔的條件
3. 看可訓練參數變多少，再和練習四的 val 正確率比一比

> 只有 val 創新高才存檔（`best.pt`），測試集留到最後才用一次；答案版最後有一格老師示範，用 `classification_report` 看每一類的表現。

**要繳交**

- 截圖 ⑤：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習五：解凍 layer4，兩組學習率微調（階段二）
for p in model.___.parameters():     # ✏️ 骨幹最後一段
    p.requires_grad = True
print('可訓練參數：', f'{count(model):,}')
opt = torch.optim.AdamW([
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(), 'lr': ___}])  # ✏️
best = 0.0                           # 目前最好的 val
for ep in range(2):
    tr, va = run(model, train_dl, opt), run(model, val_dl)
    if va[1] > ___:                  # ✏️ 比最好的還好
        best = va[1]
        torch.save(model.state_dict(), 'best.pt')
    print(f'ep{ep}  train {tr[1]:.3f}  val {va[1]:.3f}')

#### 📸 截圖 ⑤

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：可訓練參數那一行，以及兩輪的 train／val 正確率

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 可訓練參數 8,398,858 是怎麼來的？**

（在這裡寫）

**2. 階段二的 val 正確率有比階段一高嗎？骨幹的學習率為什麼要比分類頭小？**

（在這裡寫）

## AI 使用紀錄

有用 AI 就寫下來，沒用就寫「沒有使用」。

| 用了哪個 AI | 問了什麼（大概就好） | 它哪裡幫到你 | 它哪裡說錯、你怎麼發現的 |
|---|---|---|---|
| （例：ChatGPT） | （例：requires_grad 是什麼意思） |  |  |

## 繳交前檢查

1. **重跑一次**：「執行階段 › 重新啟動工作階段並執行所有儲存格」，每一格都有輸出、沒有紅字。
2. **檔名**：`深度學習_W06_學號_姓名.ipynb`（左上角點檔名就能改）。
3. **截圖**：5 張（①～⑤），每張都是整個螢幕，看得到身份指紋（左上的檔名、右上的 Google 頭貼、右下工作列的日期時間）和那一題的輸出。
4. **開放分享**：右上角「分享」→「一般存取權」改成「知道連結的任何人」→ 角色選「檢視者」→「複製連結」。
5. **自己檢查**：開一個無痕視窗貼上連結，不用登入就打得開才算成功。
6. **下載**：「檔案 › 下載 › 下載 .ipynb」。
7. **交到 Flipclass**：分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤），交到 Flipclass 作業區，期限照 Flipclass 規定。